# 09b — Inspect and prepare Spanish mobile-phone OD data

This notebook examines one day of the downloaded mobile-phone OD data and creates compact intra-Sevilla tables for later analysis.

The goal is descriptive quality control:

- separate intra-Sevilla, inbound, and outbound flows;
- inspect hourly mobility;
- save aggregated OD/origin/destination tables.

No heat effect is estimated here.

In [ ]:
from pathlib import Path

# The notebooks_v02 folder is expected to sit directly inside the repository.
# If you open Jupyter from another working directory, set PROJECT_ROOT manually.
HERE = Path.cwd().resolve()
PROJECT_ROOT = HERE.parent if HERE.name == "notebooks_v02" else HERE

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
OUTPUTS = PROJECT_ROOT / "outputs"

for folder in [DATA_RAW, DATA_INTERIM, OUTPUTS]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd

MOBILE_INTERIM = DATA_INTERIM.parent / "mobile" / "interim"
MOBILE_PROCESSED = DATA_INTERIM.parent / "mobile" / "processed"
MOBILE_PROCESSED.mkdir(parents=True, exist_ok=True)

od = pd.read_parquet(
    MOBILE_INTERIM / "sevilla_od_2024-07-15.parquet"
)

zones = gpd.read_file(
    MOBILE_INTERIM / "sevilla_districts_v2.gpkg"
)

print("OD rows:", f"{len(od):,}")
print("Sevilla districts:", len(zones))

## 1. Classify flows

In [ ]:
sevilla_ids = set(zones["id"].astype(str))

od["id_origin"] = od["id_origin"].astype(str)
od["id_destination"] = od["id_destination"].astype(str)

od["flow_type"] = "external"

od.loc[
    od["id_origin"].isin(sevilla_ids)
    & od["id_destination"].isin(sevilla_ids),
    "flow_type",
] = "intra_sevilla"

od.loc[
    od["id_origin"].isin(sevilla_ids)
    & ~od["id_destination"].isin(sevilla_ids),
    "flow_type",
] = "outbound"

od.loc[
    ~od["id_origin"].isin(sevilla_ids)
    & od["id_destination"].isin(sevilla_ids),
    "flow_type",
] = "inbound"

flow_summary = (
    od.groupby("flow_type")["n_trips"]
    .agg(rows="size", trips="sum")
)

display(flow_summary)

## 2. Aggregate intra-Sevilla flows

In [ ]:
intra = od.loc[
    od["flow_type"] == "intra_sevilla"
].copy()

intra_od = (
    intra.groupby(
        ["date", "hour", "id_origin", "id_destination"],
        as_index=False,
    )
    .agg(
        n_trips=("n_trips", "sum"),
        total_length_km=("trips_total_length_km", "sum"),
    )
)

origin_hour = (
    intra.groupby(
        ["date", "hour", "id_origin"],
        as_index=False,
    )
    .agg(
        trips_originating=("n_trips", "sum"),
        total_length_km=("trips_total_length_km", "sum"),
    )
)

destination_hour = (
    intra.groupby(
        ["date", "hour", "id_destination"],
        as_index=False,
    )
    .agg(
        trips_arriving=("n_trips", "sum"),
    )
)

print("Aggregated intra-Sevilla OD rows:", len(intra_od))

## 3. Inspect the hourly profile

In [ ]:
hourly = (
    intra.groupby("hour", as_index=False)["n_trips"]
    .sum()
)

hourly["share_pct"] = (
    hourly["n_trips"]
    / hourly["n_trips"].sum()
    * 100
)

afternoon_share = (
    hourly.loc[
        hourly["hour"].between(12, 18),
        "n_trips",
    ].sum()
    / hourly["n_trips"].sum()
    * 100
)

print(f"Share of trips from 12:00–18:00: {afternoon_share:.1f}%")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(hourly["hour"], hourly["n_trips"], marker="o")
ax.set_xlabel("Hour")
ax.set_ylabel("Estimated trips")
ax.set_title("Hourly intra-Sevilla mobility — 15 July 2024")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Save prepared tables

In [ ]:
intra_od.to_parquet(
    MOBILE_PROCESSED / "sevilla_intra_od_2024-07-15.parquet",
    index=False,
)

origin_hour.to_parquet(
    MOBILE_PROCESSED / "sevilla_origin_hour_2024-07-15.parquet",
    index=False,
)

destination_hour.to_parquet(
    MOBILE_PROCESSED / "sevilla_destination_hour_2024-07-15.parquet",
    index=False,
)

print("Saved prepared mobile-phone tables.")